In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta

np.random.seed(42)

N = 1000

job_titles = ["SAP", "Big Data", "DevOps Engineer", "Java", "Dot Net", "ETL"]
job_locations = ["Austin, TX", "Dallas, TX", "New York City, NY", "San Jose, CA"]
timezones = ["EST", "CST", "PST"]
visas = ["US Citizen", "H1B", "Green Card", "GC EAD"]

def noise(std, min_val=0.3):
    """Controlled noise: small, clipped, never negative"""
    return max(min_val, np.random.normal(0, std))

rows = []

start_date = pd.Timestamp("2023-01-02")  # Monday

for i in range(N):
    sourcing_start = start_date + pd.Timedelta(days=i * 3)

    dow = sourcing_start.dayofweek
    mon = sourcing_start.month

    # ---------------- Base deterministic signal ----------------
    dur_sourcing_submission = 1.5 + 0.35 * dow + 0.20 * mon
    dur_submission_interview = 2.0 + 0.25 * dow + 0.10 * mon
    dur_interview_duration = 1.2 + 0.10 * dow
    dur_interview_offer = 2.5 + 0.30 * dow + 0.15 * mon
    dur_offer_filled = 3.5 + 0.40 * dow + 0.20 * mon

    # ---------------- Add controlled noise ----------------
    dur_sourcing_submission += noise(0.5)
    dur_submission_interview += noise(0.7)
    dur_interview_duration += noise(0.3)
    dur_interview_offer += noise(0.8)
    dur_offer_filled += noise(1.0)

    submission = sourcing_start + timedelta(days=dur_sourcing_submission)
    interview_start = submission + timedelta(days=dur_submission_interview)
    interview_end = interview_start + timedelta(days=dur_interview_duration)
    offered = interview_end + timedelta(days=dur_interview_offer)
    filled = offered + timedelta(days=dur_offer_filled)

    # Slightly imperfect hiring outcome
    hired = "Y" if dur_offer_filled < 10 else "N"

    rows.append({
        "Job Title": np.random.choice(job_titles),
        "Job Location": np.random.choice(job_locations),
        "Job Time Zone": np.random.choice(timezones),
        "Consultant Name": f"Consultant_{i}",
        "Visa": np.random.choice(visas),
        "Consultant Location": np.random.choice(job_locations),
        "Consultant Time Zone": np.random.choice(timezones),
        "Salary ($1000)": f"${np.random.randint(95, 185)}K",
        "Relocation": "Yes" if dow < 4 else "No",
        "Submission Date": submission,
        "Joining Date": sourcing_start.date(),
        "Interview Status": "Cleared",
        "Hired": hired,
        "Sourcing Start": sourcing_start,
        "Interview Start": interview_start,
        "Interview End": interview_end,
        "Offered": offered,
        "Filled": filled
    })

df = pd.DataFrame(rows)
df.to_csv("generated_dataset.csv", index=False)

print("✅ Controlled-noise dataset.csv generated")


✅ Controlled-noise dataset.csv generated
